# Qwen 3.5 Tool-Calling Fine-Tuning with Unsloth

Open this notebook in Google Colab (with a free T4 GPU). Upload your `qwen_training_dataset.jsonl` to the sidebar.

In [ ]:
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # Use 4bit to fit on Colab T4 GPU

# 1. Load Qwen 3.5 base model (0.8B or 2B)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-2B-Instruct-bnb-4bit", # Or unsloth/Qwen3.5-0.8B-Instruct-bnb-4bit
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = None, # Specify HF token if using gated models
)


In [ ]:
# 2. Apply LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 3. Load Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Ensure the ChatML format is perfectly matched for Tool Calling.
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5", # Qwen 3.5 shares this Unsloth unified template mapping
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="qwen_training_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Setup SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase for real training loop (e.g. 300+)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 5. Export to GGUF
model.save_pretrained("qwen_lora_tool_calling")  # Save LoRA locally in Colab

# Export to GGUF specifically so you can run it via Llama.cpp / Ollama back on Mac
model.save_pretrained_gguf("qwen_tool_lora", tokenizer, quantization_method = "q4_k_m") 

# You can now download `qwen_tool_lora-unsloth.Q4_K_M.gguf` from the Colab file explorer.